# Build a small language model from scratch

A ~14M-parameter, Llama-style model trained on **SimpleStories** on a laptop GPU (RTX 3050 4 GB).
Run the cells **top to bottom**. Each section explains what the next cell does.

**Pipeline:** text → tokenizer → numbers → transformer → predicts the next token → repeat for millions of steps.

## 0. Check the GPU
Training on CPU would be ~50x slower. This must print `True`.

In [1]:
import math, os, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

torch 2.6.0+cu124 | cuda available: True
NVIDIA GeForce RTX 3050 4GB Laptop GPU


## 1. Data: download, tokenizer, token files

- **Tokenizer (BPE, 8k vocab):** splits text into ~8,000 common pieces (`"walked"` might be one token). The model only sees integer ids.
- We turn every story into ids, put `<|endoftext|>` between stories, and save one long array of `uint16` numbers (`data/train.bin`).

Skipped automatically if the files already exist.

In [2]:
from datasets import load_dataset
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
import itertools

VOCAB_SIZE = 8192
EOS = "<|endoftext|>"
os.makedirs("data", exist_ok=True)

if not os.path.exists("data/train.bin"):
    ds = load_dataset("lennart-finke/SimpleStories")
    print(ds)                       # shows split names (train / test)

    def batches(split, n=1000):
        d = ds[split]
        for i in range(0, len(d), n):
            yield d[i:i+n]["story"]

    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=[EOS],
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tok.train_from_iterator(itertools.islice(batches("train"), 200), trainer)
    tok.save("data/tokenizer.json")
    eos_id = tok.token_to_id(EOS)

    def write(name, split):
        total = 0
        with open(f"data/{name}.bin", "wb") as f:
            for texts in batches(split, 5000):
                ids = []
                for e in tok.encode_batch(texts):
                    ids.extend(e.ids); ids.append(eos_id)
                f.write(np.array(ids, dtype=np.uint16).tobytes())
                total += len(ids)
        print(f"{name}: {total/1e6:.1f}M tokens")

    write("validation", "test")
    write("train", "train")
else:
    print("data already prepared")

tok = Tokenizer.from_file("data/tokenizer.json")
enc = tok.encode("The old man walked into the forest.")
print(enc.tokens)
print(enc.ids)

C:\slm_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data already prepared
['The', 'Ġold', 'Ġman', 'Ġwalked', 'Ġinto', 'Ġthe', 'Ġforest', '.']
[285, 557, 629, 763, 507, 262, 870, 14]


## 2. Model config

Small enough for 4 GB of VRAM. `CTX` = how many tokens the model can look at at once.
`MICRO_BS × ACCUM × CTX` = tokens per optimizer step (~131k).

In [3]:
VOCAB, DIM, LAYERS, HEADS, FF, CTX = 8192, 384, 6, 6, 1024, 512
MICRO_BS, ACCUM = 16, 16          # if you get an out-of-memory error: 8 and 32
MAX_STEPS, WARMUP = 3000, 200
LR, MIN_LR, WD = 1e-3, 1e-4, 0.1
CKPT = "ckpt.pt"

dev = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if dev == "cuda" else torch.float32

## 3. The model, piece by piece

**RMSNorm** keeps numbers at a stable scale. **RoPE** tells attention *where* each token is by rotating query/key vectors.

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32 * self.w).type_as(x)

def apply_rope(x, cos, sin):          # x: (batch, heads, time, head_dim)
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([x1*cos - x2*sin, x2*cos + x1*sin], dim=-1)

**Attention:** every token looks at earlier tokens (`is_causal=True` blocks peeking at the future) and mixes in useful information.
**MLP (SwiGLU):** processes each token on its own to store/transform knowledge.

In [5]:
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(DIM, 3*DIM, bias=False)
        self.proj = nn.Linear(DIM, DIM, bias=False)
    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, HEADS, DIM//HEADS).permute(2, 0, 3, 1, 4)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, DIM))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.gate = nn.Linear(DIM, FF, bias=False)
        self.up   = nn.Linear(DIM, FF, bias=False)
        self.down = nn.Linear(FF, DIM, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

**Block** = attention + MLP, each with a residual connection (`x = x + ...`).
**GPT** = embedding → 6 blocks → norm → output scores for each of the 8,192 tokens. The output layer shares weights with the embedding (saves parameters).

In [6]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.n1, self.attn = RMSNorm(DIM), Attention()
        self.n2, self.mlp = RMSNorm(DIM), MLP()
    def forward(self, x, cos, sin):
        x = x + self.attn(self.n1(x), cos, sin)
        return x + self.mlp(self.n2(x))

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, DIM)
        self.blocks = nn.ModuleList(Block() for _ in range(LAYERS))
        self.norm = RMSNorm(DIM)
        self.head = nn.Linear(DIM, VOCAB, bias=False)
        self.head.weight = self.emb.weight
        hd = DIM // HEADS
        inv = 1.0 / (10000 ** (torch.arange(0, hd, 2).float() / hd))
        f = torch.outer(torch.arange(CTX).float(), inv)
        self.register_buffer("cos", f.cos(), persistent=False)
        self.register_buffer("sin", f.sin(), persistent=False)
        self.apply(self._init)
        for n, p in self.named_parameters():
            if n.endswith("proj.weight") or n.endswith("down.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2*LAYERS))
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        T = idx.shape[1]
        x = self.emb(idx)
        cos, sin = self.cos[:T], self.sin[:T]
        for b in self.blocks:
            x = b(x, cos, sin)
        logits = self.head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, VOCAB), targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, n=200, temp=0.8, top_k=50):
        for _ in range(n):
            logits, _ = self(idx[:, -CTX:])
            logits = logits[:, -1].float() / temp
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")
            idx = torch.cat([idx, torch.multinomial(F.softmax(logits, -1), 1)], dim=1)
        return idx

model = GPT().to(dev)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters on {dev}")

13.8M parameters on cuda


## 4. Batches, learning rate, sampling

- **Batch:** random chunks of 512 tokens; the target `y` is the same chunk shifted by one (predict the next token at every position).
- **Learning rate:** ramps up for 200 steps, then slowly decays (cosine).

In [7]:
data = {}
def get_batch(split):
    if split not in data:
        data[split] = np.memmap(f"data/{split}.bin", dtype=np.uint16, mode="r")
    d = data[split]
    ix = np.random.randint(0, len(d) - CTX - 1, MICRO_BS)
    x = torch.stack([torch.from_numpy(d[i:i+CTX].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(d[i+1:i+1+CTX].astype(np.int64)) for i in ix])
    return x.to(dev), y.to(dev)

def get_lr(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    p = (step - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return MIN_LR + 0.5 * (LR - MIN_LR) * (1 + math.cos(math.pi * p))

@torch.no_grad()
def estimate_loss(iters=50):
    model.eval(); out = {}
    for split in ("train", "validation"):
        ls = []
        for _ in range(iters):
            x, y = get_batch(split)
            with torch.autocast(dev, dtype=dtype, enabled=dev == "cuda"):
                ls.append(model(x, y)[1].item())
        out[split] = round(sum(ls)/len(ls), 3)
    model.train(); return out

def sample(prompt="Once upon a time", n=200, temp=0.8):
    model.eval()
    ids = torch.tensor([tok.encode(prompt).ids], device=dev)
    with torch.autocast(dev, dtype=dtype, enabled=dev == "cuda"):
        out = model.generate(ids, n=n, temp=temp)
    model.train()
    return tok.decode(out[0].tolist())

# before training: random weights -> gibberish
print(sample("The old man walked into the", n=40))

The old man walked into the stands flick capturing shining NotButration sentairs m desp mouth smarter surge triump siblings siblings kissed peaceful forgiveThThThCold soaked silver forgive forgiveairs comet cometicing gave artifact jo jo walks planned stopping atmos


## 5. Train

Each step: run several mini-batches, add up their gradients, then update the weights once.
Watch `loss` go down (starts near 9, should reach ~2 or lower) and `tok/s` (should be tens of thousands on GPU).

- **Stop anytime** with the ■ interrupt button. A checkpoint is saved and **re-running this cell resumes**.
- Do a quick test first: set `MAX_STEPS = 200` in section 2, run, then delete `ckpt.pt` before the real run.
- Keep the laptop plugged in and awake. For overnight runs, the terminal (`python train.py`) is safer than a notebook.

In [8]:
decay   = [p for p in model.parameters() if p.dim() >= 2]
nodecay = [p for p in model.parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": WD},
                         {"params": nodecay, "weight_decay": 0.0}],
                        lr=LR, betas=(0.9, 0.95), fused=dev == "cuda")

start = 0
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=dev)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); start = ck["step"]
    print("resumed from step", start)

def save(step):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "step": step}, CKPT)

model.train()
t0, step = time.time(), start
try:
    for step in range(start, MAX_STEPS):
        for g in opt.param_groups: g["lr"] = get_lr(step)
        for _ in range(ACCUM):
            x, y = get_batch("train")
            with torch.autocast(dev, dtype=dtype, enabled=dev == "cuda"):
                _, loss = model(x, y)
            (loss / ACCUM).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); opt.zero_grad(set_to_none=True)

        if step % 20 == 0:
            if dev == "cuda": torch.cuda.synchronize()
            dt = time.time() - t0; t0 = time.time()
            tps = MICRO_BS * CTX * ACCUM * (20 if step > start else 1) / dt
            print(f"step {step} | loss {loss.item():.3f} | {tps:,.0f} tok/s")
        if step > start and step % 250 == 0:
            print("EVAL", estimate_loss())
        if step > start and step % 500 == 0:
            print("--- sample ---\n" + sample() + "\n--------------")
            save(step + 1)
    save(MAX_STEPS); print("done")
except KeyboardInterrupt:
    save(step); print(f"\nstopped, checkpoint saved at step {step}")

resumed from step 1095
step 1100 | loss 2.132 | 107,152 tok/s
step 1120 | loss 1.961 | 32,354 tok/s
step 1140 | loss 1.955 | 32,346 tok/s
step 1160 | loss 2.008 | 32,354 tok/s
step 1180 | loss 1.960 | 32,354 tok/s
step 1200 | loss 1.852 | 32,356 tok/s
step 1220 | loss 1.922 | 32,359 tok/s
step 1240 | loss 1.983 | 32,358 tok/s
EVAL {'train': 1.951, 'validation': 1.966}
step 1260 | loss 1.954 | 29,293 tok/s
step 1280 | loss 1.980 | 32,350 tok/s
step 1300 | loss 2.011 | 32,357 tok/s
step 1320 | loss 1.890 | 32,358 tok/s
step 1340 | loss 1.894 | 32,295 tok/s
step 1360 | loss 1.825 | 32,350 tok/s
step 1380 | loss 1.917 | 32,350 tok/s
step 1400 | loss 1.905 | 32,356 tok/s
step 1420 | loss 1.983 | 32,374 tok/s
step 1440 | loss 1.945 | 32,374 tok/s
step 1460 | loss 1.864 | 32,368 tok/s
step 1480 | loss 1.905 | 28,254 tok/s
step 1500 | loss 1.883 | 32,379 tok/s
EVAL {'train': 1.914, 'validation': 1.91}
--- sample ---
Once upon a time, there was a land full of light and color.

A girl named Lena

## 6. Write stories
Change the prompt and `temp` (0.6 = safe/repetitive, 1.0 = creative/messy).

In [9]:
print(sample("The old man walked into the", n=250, temp=0.8))

The old man walked into the classroom, smiling knowingly. His face lit up as he handed him the golden key. "This key opens the door to your dreams," he said. The boy hesitated, feeling shy. He thought about the stories he had heard. But he knew he needed to see them.

He entered the room, and the door was closed. "Where does the key go?" he asked, feeling a bit scared. The old man sat beside him. "To find the door, you must show me your courage," he said. "You must find your courage." The boy took a deep breath and remembered his dreams of flying. He felt a spark of courage inside him.

He closed his eyes and took a deep breath. The room began to shake, and he felt weightless. The door opened wider, and he stepped back into the light. Outside, the sun was shining bright. He rushed to school, heart pounding. He thought about what he had seen. Was it a dream? He wanted to believe in his dreams.

When he returned to school, he told his friends. "I found a key!" he said. They listened clos

In [10]:
torch.save({
    "model": model.state_dict(),
    "config": dict(VOCAB=VOCAB, DIM=DIM, LAYERS=LAYERS, HEADS=HEADS, FF=FF, CTX=CTX),
}, "slm_final.pt")
print("saved", os.path.getsize("slm_final.pt") / 1e6, "MB")

saved 55.086898 MB


To use it laterimport torch
from tokenizers import Tokenizer
from train import GPT, sample, dev

tok = Tokenizer.from_file("data/tokenizer.json")
model = GPT().to(dev)
model.load_state_dict(torch.load("slm_final.pt", map_location=dev)["model"])
print(sample(model, tok, "The old man walked into the"))

In [11]:
# import torch
# from tokenizers import Tokenizer
# from train import GPT, sample, dev

# tok = Tokenizer.from_file("data/tokenizer.json")
# model = GPT().to(dev)
# model.load_state_dict(torch.load("slm_final.pt", map_location=dev)["model"])
# print(sample(model, tok, "The old man walked into the"))